In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 — Bronze Layer: Raw Ingestion
# MAGIC
# MAGIC **Pipeline**: Unity Catalog Volume CSV → `bronze_facilities_raw` Delta Table
# MAGIC
# MAGIC This layer is the **immutable raw record**. The only operations are:
# MAGIC - Column name sanitisation (spaces/special chars → snake_case)
# MAGIC - Metadata audit columns (_ingested_at, _row_hash, etc.)
# MAGIC - Quality gate with MLflow logging
 
# COMMAND ----------
# MAGIC %md ## 0. Imports & Setup
 
# COMMAND ----------
 
import re
from datetime import datetime,UTC
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType
 
spark = SparkSession.builder.getOrCreate()
# spark.conf.set("spark.sql.adaptive.enabled", "true")
print(f"Spark version : {spark.version}")
print(f"Run timestamp : {datetime.now(UTC).isoformat()}")
 
# COMMAND ----------
# MAGIC %md ## 1. Configuration
 
# COMMAND ----------
 
class BronzeConfig:
    VOLUME_PATH      = "/Volumes/virtue_foundation/ghana/volume"
    SOURCE_CSV       = f"{VOLUME_PATH}/Virtue Foundation Ghana v0.3 - Sheet1.csv"
 
    CATALOG          = "virtue_foundation"
    SCHEMA           = "ghana"
    BRONZE_TABLE     = "virtue_foundation.ghana.bronze_facilities_raw"
 
    COUNTRY_SCOPE    = "GH"
    DATASET_VERSION  = "v0.3"
    SOURCE_FILE_NAME = "Virtue_Foundation_Ghana_v0.3.csv"
 
    CSV_OPTIONS = {
        "header"                    : "true",
        "multiLine"                 : "true",
        "escape"                    : '"',
        "quote"                     : '"',
        "inferSchema"               : "false",   # all STRING in Bronze
        "encoding"                  : "UTF-8",
        "ignoreLeadingWhiteSpace"   : "true",
        "ignoreTrailingWhiteSpace"  : "true",
        "mode"                      : "PERMISSIVE",
    }
 
cfg = BronzeConfig()
print(f"Source  : {cfg.SOURCE_CSV}")
print(f"Target  : {cfg.BRONZE_TABLE}")
 
# COMMAND ----------
# MAGIC %md ## 2. Ensure Catalog & Schema Exist
 
# COMMAND ----------
 
spark.sql(f"CREATE CATALOG  IF NOT EXISTS {cfg.CATALOG}")
spark.sql(f"USE CATALOG {cfg.CATALOG}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {cfg.SCHEMA}")
print(f"✅  {cfg.CATALOG}.{cfg.SCHEMA} ready")
 
# COMMAND ----------
# MAGIC %md ## 3. Read Raw CSV
 
# COMMAND ----------
 
raw_df = spark.read.options(**cfg.CSV_OPTIONS).csv(cfg.SOURCE_CSV)
print(f"Raw rows    : {raw_df.count():,}")
print(f"Raw columns : {len(raw_df.columns)}")
 
# COMMAND ----------
# MAGIC %md ## 4. Column Name Sanitisation
# MAGIC
# MAGIC Fixes: `mongo DB` → `mongo_db`, camelCase → lowercase, spaces → underscores.
# MAGIC Verified against actual CSV headers from the dataset.
 
# COMMAND ----------
 
def sanitise_col_name(name: str) -> str:
    """Convert any header to safe lowercase snake_case."""
    name = name.strip()
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.lower().strip("_")
 
rename_map = {
    col: sanitise_col_name(col)
    for col in raw_df.columns
    if sanitise_col_name(col) != col
}
 
print(f"Columns renamed ({len(rename_map)}):")
for old, new in rename_map.items():
    print(f"  '{old}'  →  '{new}'")
 
for old, new in rename_map.items():
    raw_df = raw_df.withColumnRenamed(old, new)
 
# COMMAND ----------
# MAGIC %md ## 5. Add Audit Columns
 
# COMMAND ----------
 
# hash_cols = [F.col(c) for c in raw_df.columns if not c.startswith("_")]
# hash_expr = F.sha2(F.concat_ws("||", *hash_cols), 256)
hash_cols = [F.coalesce(F.col(c).cast("string"), F.lit("")) for c in raw_df.columns if not c.startswith("_")]
hash_expr = F.sha2(F.concat_ws("||", *hash_cols), 256)
 
bronze_df = (
    raw_df
    .withColumn("ingested_at",     F.current_timestamp())
    .withColumn("source_file",     F.lit(cfg.SOURCE_FILE_NAME))
    .withColumn("dataset_version", F.lit(cfg.DATASET_VERSION))
    .withColumn("country_scope",   F.lit(cfg.COUNTRY_SCOPE))
    .withColumn("row_hash",        hash_expr)
)
 
# if "_corrupt_record" not in bronze_df.columns:
#     bronze_df = bronze_df.withColumn("_corrupt_record", F.lit(None).cast(StringType()))
 
print(f"Bronze columns (incl. audit) : {len(bronze_df.columns)}")
 
# COMMAND ----------
# MAGIC %md ## 6. Quality Gate
 
# COMMAND ----------
 
total_rows     = bronze_df.count()
# corrupt_rows   = bronze_df.filter(F.col("_corrupt_record").isNotNull()).count()
null_unique_id = bronze_df.filter(
    F.col("unique_id").isNull() | (F.trim(F.col("unique_id")) == "")
).count()
null_name = bronze_df.filter(
    F.col("name").isNull() | (F.trim(F.col("name")) == "")
).count()
 
print("=" * 60)
print("BRONZE QUALITY GATE")
print("=" * 60)
print(f"Total rows              : {total_rows:>6,}")
# print(f"Corrupt rows            : {corrupt_rows:>6,}")
print(f"Rows with null unique_id: {null_unique_id:>6,}")
print(f"Rows with null name     : {null_name:>6,}")
 
# if corrupt_rows > 0:
#     print("\nCorrupt row samples:")
#     bronze_df.filter(F.col("_corrupt_record").isNotNull()) \
#              .select("name", "unique_id", "_corrupt_record").show(5, truncate=120)
 
try:
    import mlflow
    with mlflow.start_run(run_name="01_bronze_ingestion"):
        mlflow.log_param("source_file",     cfg.SOURCE_FILE_NAME)
        mlflow.log_param("dataset_version", cfg.DATASET_VERSION)
        mlflow.log_metric("total_rows",     total_rows)
        # mlflow.log_metric("corrupt_rows",   corrupt_rows)
        mlflow.log_metric("null_unique_id", null_unique_id)
        mlflow.log_param("bronze_table",    cfg.BRONZE_TABLE)
    print("MLflow logged ✅")
except Exception as e:
    print(f"MLflow skipped: {e}")
 
# COMMAND ----------
# MAGIC %md ## 7. Write Bronze Delta Table
 
# COMMAND ----------
 
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact",   "true")
    .saveAsTable(cfg.BRONZE_TABLE)
)
 
readback = spark.table(cfg.BRONZE_TABLE).count()
assert readback == total_rows, f"Row count mismatch: wrote {total_rows}, read {readback}"
print(f"\n✅  Bronze table: {cfg.BRONZE_TABLE}")
print(f"    Rows    : {readback:,}")
print(f"    Columns : {len(bronze_df.columns)}")
 
# COMMAND ----------
# MAGIC %md ## 8. Column Inventory
 
# COMMAND ----------
 
print("COLUMN INVENTORY (all 47 columns — reference for downstream notebooks):")
print(f"{'#':<4} {'column_name':<45} {'sample_value'}")
print("─" * 80)
verify = spark.table(cfg.BRONZE_TABLE)
for i, (c, _) in enumerate(verify.dtypes, 1):
    dtype = dict(verify.dtypes)[c]

    if dtype == "string":
        condition = (
            F.col(c).isNotNull() &
            (F.trim(F.col(c)) != "") &
            (F.lower(F.col(c)) != "null")
        )
    else:
        condition = F.col(c).isNotNull()

    sample = verify.select(c).filter(condition).limit(1).collect()
    sv = str(sample[0][0])[:65] if sample else "—"
    print(f"  {i:>2}. {c:<45} {sv}")
 
# COMMAND ----------
display(
    spark.table(cfg.BRONZE_TABLE)
    .select("unique_id", "name", "organization_type", "facilitytypeid",
            "address_city", "address_stateorregion",
            "specialties", "capability", "procedure", "equipment")
    .limit(10)
)

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.bronze_facilities_raw LIMIT 250

In [0]:
%sql
DESCRIBE TABLE EXTENDED virtue_foundation.ghana.bronze_facilities_raw

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.bronze_facilities_raw